# Humanoid retargeting — compare your Layer-2 run with the paper

`L2_eval.sh` computes every table row from scratch: the method retargets the test clips, MuJoCo plays the result with the TWIST tracker, and the scorer writes one row per clip to `release_data/retargeting/L2_outputs/reports/`. This notebook recomputes the table from your rows with the same code as `L1_tables.ipynb` and prints **paper vs L1 (released per-clip results of the paper) vs L2 (your run)**.

Rows you have not run yet are listed as `pending`. The column **L3** scores a checkpoint you trained with `L3_train.sh`, evaluated with `WMM_CKPT=<checkpoint> WMM_TAG=_l3 L2_eval.sh wmm_<setting>` (WMM rows only).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
sys.path.insert(0, str(HERE))
import l1_retargeting as L

L1 = L.l1_dir(HERE)
L2 = L.l2_reports_dir(HERE)
L.verify_checksums(L1)
sids = L.test_sids(L1)
print("L1 reports:", L1 / "reports")
print("L2 reports:", L2)

In [ ]:
def load_reports(tag="", methods=L.METHODS):
    seq = {
        key: L.load_per_sequence(L2, stem + tag, sids)
        for key, stem in L.RUNS.items()
        if key[0] in methods and (L2 / f"{stem}{tag}_mjc_rollout_report_{L.N_FIRST}.xlsx").exists()
    }
    return seq, {key: L.recompute(df) for key, df in seq.items()}


l1_seq = {key: L.load_per_sequence(L1 / "reports", stem, sids) for key, stem in L.RUNS.items()}
l1 = {key: L.recompute(df) for key, df in l1_seq.items()}
l2_seq, l2 = load_reports()
l3_seq, l3 = load_reports("_l3", methods=["WMM"])
print(f"L2 rows available: {len(l2)}/{len(L.RUNS)}; L3 rows available: {len(l3)}/3")
for key in L.RUNS:
    if key not in l2:
        print("  L2 pending:", key)

## 1. Paper vs L1 vs L2 (and L3 for WMM), cell by cell (SR-seq, gerr, lerr)

In [ ]:
rows = []
for key, cells in L.PAPER.items():
    for metric, paper_str in cells.items():
        if metric == "FPS":
            continue
        paper = float(paper_str)
        l2_val = l2[key][metric] if key in l2 else np.nan
        l3_val = l3[key][metric] if key in l3 else np.nan
        rows.append({
            "method": key[0], "setting": key[1], "metric": metric,
            "paper": paper_str,
            "L1": l1[key][metric],
            "L2": l2_val,
            "L2 - paper": l2_val - paper,
            "rel": (l2_val - paper) / paper,
            "L3": "-" if key[0] != "WMM" else "pending" if key not in l3 else f"{l3_val:.4f}",
            "L3 - paper": "-" if key[0] != "WMM" else "pending" if key not in l3 else f"{l3_val - paper:.4f}",
        })
compare = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print(compare.to_string(index=False, na_rep="pending"))

In [ ]:
fps = pd.DataFrame({
    m: {
        "paper (reported)": L.PAPER[(m, "clean")]["FPS"],
        "your run (information only)": f"{l2[(m, 'clean')]['FPS']:.2f}" if (m, "clean") in l2 else "pending",
    }
    for m in L.METHODS
}).T
print("Clean-setting speed, frames per second:")
fps

## 2. Your table in the paper's layout

In [ ]:
if len(l2) == len(L.RUNS):
    table = L.paper_layout(l2)
    table[("Clean (30 FPS)", "FPS")] = [L.PAPER[(m, "clean")]["FPS"] + "*" for m in L.METHODS]
    print(table.to_string())
    print("* FPS as reported in the paper")
else:
    print("table incomplete; finished rows are listed above")

## 3. Clip by clip: your run vs the paper's per-clip results

For each finished row: the share of clips with the same success/fall outcome, and the correlation of the per-clip local error `lerr` between the paper's run (L1) and yours (L2).

In [ ]:
agree = pd.DataFrame({
    key: {
        "same SR outcome": float((l1_seq[key]["SR-seq"] == l2_seq[key]["SR-seq"]).mean()),
        "lerr corr": float(np.corrcoef(l1_seq[key]["lerr"], l2_seq[key]["lerr"])[0, 1]),
    }
    for key in l2_seq
}).T
agree

In [ ]:
keys = list(l2_seq)
if keys:
    fig, axes = plt.subplots(1, len(keys), figsize=(3.2 * len(keys), 3.2), squeeze=False)
    for ax, key in zip(axes[0], keys):
        ax.scatter(100 * l1_seq[key]["lerr"], 100 * l2_seq[key]["lerr"], s=3, alpha=0.4)
        lim = 100 * max(l1_seq[key]["lerr"].quantile(0.99), l2_seq[key]["lerr"].quantile(0.99))
        ax.plot([0, lim], [0, lim], "k--", lw=0.8)
        ax.set_xlim(0, lim)
        ax.set_ylim(0, lim)
        ax.set_title(f"{key[0]} {key[1]}", fontsize=9)
        ax.set_xlabel("paper lerr (cm)")
    axes[0][0].set_ylabel("your lerr (cm)")
    fig.tight_layout()
    plt.show()